In [0]:
from pyspark.sql.functions import *
import pyspark.sql.functions as F

In [0]:
catalog ='ecommerce'
schema ='raw'
volume ='raw_data'

## Data Cleaning Data Modelling

### Customer Table

In [0]:
brz_customers = spark.table(f"{catalog}.{schema}.brz_customers")
display(brz_customers.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- signup_channel: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- is_active: boolean (nullable = true)



In [0]:
slv_customers = (brz_customers.withColumn("phone",F.regexp_extract(F.col("phone").cast("string"),r"(\d{10})$",1))
                    .withColumn("phone",F.when((F.col("phone") == "0") |(F.col("phone") == ""),None).otherwise
                                (F.col("phone")))
                        .dropDuplicates(["customer_id"]))

display(slv_customers.limit(10))

customer_id,customer_name,email,phone,city,state,pincode,signup_date,signup_channel,customer_segment,is_active
CST0001,Veda Ratta,shahvasana@example.org,5715364107,Madanapalle,Tamil Nadu,128026,2024-04-03,email,Premium,true
CST0002,Onveer More,vwalla@example.org,null,Bharatpur,Arunachal Pradesh,997695,2025-02-22,social,Premium,true
CST0003,Vedhika Goyal,christopheragate@example.org,1365957295,Kota,Himachal Pradesh,563270,2026-02-27,referral,Regular,true
CST0004,Owen Aurora,vivaansrinivasan@example.com,2478471358,Kota,Goa,160147,2024-12-10,email,VIP,true
CST0005,Ekalinga Sarkar,banjeetrama@example.net,8367567096,Ambala,Nagaland,333674,2026-01-21,paid_search,Regular,true
CST0006,Yachana Choudhry,saumya43@example.com,6958813887,Bhiwandi,Sikkim,184401,2025-11-15,email,Premium,true
CST0007,Ayaan Bose,urvashioak@example.net,6036415727,Vasai-Virar,Arunachal Pradesh,19030,2026-06-23,organic,Premium,true
CST0008,Urvi Ravi,hayeryash@example.org,1269280144,Purnia,Jharkhand,847193,2025-11-29,organic,VIP,true
CST0009,Baghyawati Wadhwa,fariq47@example.com,7652821108,Berhampur,Kerala,916480,2025-09-06,organic,Regular,true
CST0010,Dayamai Bhat,jalsabhatia@example.com,7668993603,Danapur,Meghalaya,778427,2024-07-27,paid_search,VIP,true


In [0]:
slv_customers.write\
                .mode('overwrite')\
                    .format('delta')\
                    .option('mergeschema','true')\
                        .saveAsTable(f"{catalog}.{schema}.slv_customers")   

In [0]:
print("Total no of Customers in silver table:",slv_customers.count())

Total no of Customers in silver table: 5000


### Products Table

In [0]:
brz_products = spark.table(f"{catalog}.{schema}.brz_products")
display(brz_products.printSchema())


root
 |-- product_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- review_count: integer (nullable = true)



In [0]:
display(brz_products.limit(5))

product_id,brand,category,sub_category,price,rating,review_count
PRD0001,"Konda, Keer and Dhawan",Sports & Fitness,Football,442722.82,3.3,4355
PRD0002,"Sangha, Dhaliwal and Sachdeva",Books,Biography,200708.22,3.6,675
PRD0003,Sanghvi-Nadig,Clothing,Shirts,220983.37,1.3,4597
PRD0004,"Kumar, Bail and Gupta",Grocery,Staples,480235.53,3.7,1248
PRD0005,"Nair, Behl and Uppal",Clothing,Sarees,225689.16,4.3,4622


In [0]:
slv_products = brz_products.dropDuplicates(['product_id'])

In [0]:
slv_products.write\
            .mode('overwrite')\
            .option('mergeschema','true')\
            .format('delta')\
            .saveAsTable(f"{catalog}.{schema}.slv_products")

In [0]:
print("Total no of Products in silver table:",slv_products.count())

Total no of Products in silver table: 1000


### Orders Table

In [0]:
brz_orders = spark.table(f"{catalog}.{schema}.brz_orders")
display(brz_orders.limit(5))

order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status
ORD00001,CST1479,PRD0315,2025-07-03,6,67666.1,UPI,Shipped
ORD00002,CST0093,PRD0289,2025-10-15,6,484419.64,Net Banking,Delivered
ORD00003,CST0386,PRD0509,2025-01-31,1,449794.29,Net Banking,Processing
ORD00004,CST2782,PRD0237,2025-09-30,4,45392.84,Debit Card,Shipped
ORD00005,CST3041,PRD0156,2025-10-27,10,259678.66,UPI,Cancelled


In [0]:
slv_orders = brz_orders.dropDuplicates(['order_id'])

In [0]:
slv_orders.write\
        .mode('overwrite')\
        .option('mergeschema','true')\
            .format('delta')\
            .saveAsTable(f'{catalog}.{schema}.slv_orders')

In [0]:
print("Total no of Orders in silver table:",slv_orders.count())

Total no of Orders in silver table: 10000


### Delivery Table

In [0]:
brz_delivery = spark.table(f'{catalog}.{schema}.brz_delivery')
display(brz_delivery.limit(5))

delivery_id,order_id,order_date,estimated_delivery_date,delivered_date,delivery_partner,delivery_status
DLV00001,ORD00001,2025-07-03,2025-07-05,null,Ecom Express,Failed
DLV00002,ORD00002,2025-10-15,2025-10-21,null,XpressBees,Delayed
DLV00003,ORD00003,2025-01-31,2025-02-02,null,DTDC,Failed
DLV00004,ORD00004,2025-09-30,2025-10-06,2025-10-08,Delhivery,Delivered
DLV00005,ORD00005,2025-10-27,2025-11-01,null,Delhivery,Out for Delivery


In [0]:
display(brz_delivery.printSchema(5))

root
 |-- delivery_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- estimated_delivery_date: date (nullable = true)
 |-- delivered_date: date (nullable = true)
 |-- delivery_partner: string (nullable = true)
 |-- delivery_status: string (nullable = true)



In [0]:
slv_delivery = brz_delivery.dropDuplicates(['delivery_id'])

In [0]:
slv_delivery.select([
                F.count(F.when(F.col(c).isNull(),c)).alias(c)
                for c in slv_delivery.columns
]).display()

delivery_id,order_id,order_date,estimated_delivery_date,delivered_date,delivery_partner,delivery_status
0,0,0,0,8076,0,0


In [0]:
slv_delivery.filter(
                (F.col('delivery_status')== 'Delivered') & 
                (F.col('delivered_date').isNull())
).display() 

delivery_id,order_id,order_date,estimated_delivery_date,delivered_date,delivery_partner,delivery_status


In [0]:
slv_delivery.write\
            .mode('overwrite')\
                .option('mergeschema','true')\
                    .format('delta')\
                        .saveAsTable(f'{catalog}.{schema}.slv_delivery')

In [0]:
print("Total no of Deliveries in silver table:",slv_delivery.count())

Total no of Deliveries in silver table: 10000


### Apply cdc using merge

In [0]:
display(
    spark.table("ecommerce.raw.slv_orders")
    .filter(F.col("order_id") == "ORD90001")
)
                


order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status


### Merge insert into original silver layer

In [0]:
from delta.tables import DeltaTable

### Insert Operation

In [0]:
target = DeltaTable.forName(spark,
                            "ecommerce.raw.slv_orders")

cdc_insert = (
    spark.table("ecommerce.raw.brz_orders_cdc")
    .filter(
        (F.col("order_id") == "ORD90001") &
        (F.col("operation") == "I")
    )
)

(target.alias('t')
        .merge(
            cdc_insert.alias('s'),
            't.order_id = s.order_id'
    )
      .whenNotMatchedInsert(
          values={
              "order_id": "s.order_id",
            "customer_id": "s.customer_id",
            "product_id": "s.product_id",
            "order_date": "CAST(s.event_ts AS DATE)",
            "qty": "s.quantity",
            "unit_price": "s.unit_price",
            "payment_method": "NULL",
            "order_status": "s.status"
          }
      )
        .execute()
      )      



DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table("ecommerce.raw.slv_orders")
    .filter(F.col("order_id") == "ORD90001")
)

order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status
ORD90001,CUST00001,PROD00001,2026-09-05,2,1500.0,null,Shipped


### Update operation

In [0]:
cdc_update = (
    spark.table("ecommerce.raw.brz_orders_cdc")
    .filter(
        (F.col("order_id") == "ORD90001") &
        (F.col("operation") == "U")
    )
)


( 
 target.alias('t')
        .merge(
            cdc_update.alias('s'),
             "t.order_id = s.order_id"
        )
    .whenMatchedUpdate(
        set ={
             "customer_id": "s.customer_id",
            "product_id": "s.product_id",
            "qty": "s.quantity",
            "unit_price": "s.unit_price",
            "order_status": "s.status"
        }
).execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table("ecommerce.raw.slv_orders")
    .filter(F.col("order_id") == "ORD90001")
)

order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status
ORD90001,CUST00001,PROD00001,2026-09-05,2,1500.0,null,Delivered


### Automating using foreacbatch() cdc operatons

#### Creating cdc merge functions

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window



def merge_cdc(batch_df, batch_id):

    target = DeltaTable.forName(
        spark,
        "ecommerce.raw.slv_orders"
    )

    # Process events in chronological order
    events = (
        batch_df
        .orderBy("event_ts")
        .collect()
    )

    for row in events:

        source_df = spark.createDataFrame(
            [row],
            batch_df.schema
        )

        (
            target.alias("t")
            .merge(
                source_df.alias("s"),
                "t.order_id = s.order_id"
            )
            .whenMatchedUpdate(
                condition="s.operation = 'U'",
                set={
                    "customer_id": "s.customer_id",
                    "product_id": "s.product_id",
                    "qty": "s.quantity",
                    "unit_price": "s.unit_price",
                    "order_status": "s.status"
                }
            )
            .whenNotMatchedInsert(
                condition="s.operation = 'I'",
                values={
                    "order_id": "s.order_id",
                    "customer_id": "s.customer_id",
                    "product_id": "s.product_id",
                    "order_date": "CAST(s.event_ts AS DATE)",
                    "qty": "s.quantity",
                    "unit_price": "s.unit_price",
                    "payment_method": "NULL",
                    "order_status": "s.status"
                }
            )
            .execute()
        )

#### Reading cdc brz table as stream

#### Starting automatic cdc pipeline

In [0]:
cdc_query = (
    cdc_stream.writeStream
        .foreachBatch(merge_cdc)
        .option(
            "checkpointLocation",
            "/Volumes/ecommerce/raw/raw_data/checkpoints/cdc_merge"
        )
        .trigger(availableNow=True)
        .start()
)

In [0]:
%sql
SELECT *
FROM ecommerce.raw.slv_orders
WHERE order_id = 'ORD999999';

order_id,customer_id,product_id,order_date,qty,unit_price,payment_method,order_status
ORD999999,CUST00001,PROD00001,2026-09-05,3,1500.0,null,Delivered
